In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu121

In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA GeForce GTX 1060 with Max-Q Design


In [2]:
%pip install faster-whisper transformers accelerate datasets jiwer noisereduce soundfile

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------- ----------- 0.8/1.1 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 6.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.8 MB 10.7 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.8 MB 13.3 MB/s eta 0:00:01
   ---------------------- ----------------- 7.3/12.8 MB 11.9 MB/s eta 0:00:01
   -------------------------------- ------- 10.5/12.8 MB 12.8 MB/s eta 0:00:01
   ------------------------------------- -- 12.1/12.8 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 11.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ---------------------------------------- 604.4/604.4 kB 7.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.2 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [3]:
%pip install "datasets<4"

  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    Found existing installation: datasets 5.1.0
    Uninstalling datasets-5.1.0:
      Successfully uninstalled datasets-5.1.0
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install "transformers==4.46.3" "huggingface-hub<1.0"

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/10.0 MB 8.4 MB/s eta 0:00:02
   ------ --------------------------------- 1.6/10.0 MB 4.4 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/10.0 MB 4.1 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/10.0 MB 5.0 MB/s eta 0:00:02
   -------------------- ------------------- 5.2/10.0 MB 5.4 MB/s eta 0:00:01
   -------------------- ------------------- 5.2/10.0 MB 5.4 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/10.0 MB 5.0 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.0 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 5.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 566.4/566.4 kB 6.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------

  You can safely remove it manually.


In [9]:
%pip install "transformers==4.46.3" "tokenizers>=0.20,<0.21" "huggingface-hub<1.0"

Note: you may need to restart the kernel to use updated packages.


In [1]:
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.is_available())

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.46.3 2.5.1+cu121 True


In [2]:
import io, time, tarfile, unicodedata, numpy as np, pandas as pd, soundfile as sf
from huggingface_hub import hf_hub_download

tsv = hf_hub_download("google/fleurs", "data/hi_in/test.tsv", repo_type="dataset")
tgz = hf_hub_download("google/fleurs", "data/hi_in/audio/test.tar.gz", repo_type="dataset")
refs_by_file = {}
for line in open(tsv, encoding="utf-8"):
    p = line.rstrip("\n").split("\t")
    refs_by_file[p[1]] = p[3]
clips = []
with tarfile.open(tgz) as tf:
    for m in tf:
        name = m.name.split("/")[-1]
        if name in refs_by_file:
            a, sr = sf.read(io.BytesIO(tf.extractfile(m).read()), dtype="float32")
            clips.append({"audio": a, "ref": refs_by_file[name]})
            if len(clips) == 100:
                break
print(len(clips), sr)

def norm(t):
    t = unicodedata.normalize("NFC", t.lower())
    t = "".join(c for c in t if unicodedata.category(c)[0] not in ("P", "S"))
    return " ".join(t.split())

def edit(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def rate(refs, hyps, char=False):
    e = n = 0
    for r, h in zip(refs, hyps):
        r, h = norm(r), norm(h)
        r, h = (r.replace(" ", ""), h.replace(" ", "")) if char else (r.split(), h.split())
        e += edit(r, h)
        n += len(r)
    return e / n

refs = [c["ref"] for c in clips]
dur = sum(len(c["audio"]) / 16000 for c in clips)

100 16000


In [4]:
from faster_whisper import WhisperModel
m = WhisperModel("base", device="cpu", compute_type="int8")

def transcribe(items):
    outs, times = [], []
    for c in items:
        t = time.time()
        segs, _ = m.transcribe(c["audio"], language="hi", beam_size=1)
        outs.append(" ".join(s.text for s in segs))
        times.append(time.time() - t)
    return outs, times

hyp_clean, t_clean = transcribe(clips)
print("clean WER", rate(refs, hyp_clean), "CER", rate(refs, hyp_clean, True), "RTF", sum(t_clean) / dur)
pd.DataFrame({"ref": refs, "hyp": hyp_clean}).to_csv("asr_baseline.csv", index=False)

config.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

E:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sankalp\.cache\huggingface\hub\models--Systran--faster-whisper-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.bin:   0%|          | 0.00/145M [00:00<?, ?B/s]

clean WER 1.0532612890775763 CER 1.059751972942503 RTF 0.27160069064448594


In [5]:
%pip install -q noisereduce
import noisereduce as nr

rng = np.random.default_rng(0)

def add_noise(x, snr=10):
    p = np.mean(x ** 2)
    return x + rng.standard_normal(len(x)).astype(np.float32) * np.sqrt(p / 10 ** (snr / 10))

N = 50
sub = clips[:N]
noisy = [{"audio": add_noise(c["audio"]), "ref": c["ref"]} for c in sub]

t0 = time.time()
den = [{"audio": nr.reduce_noise(y=c["audio"], sr=16000).astype(np.float32), "ref": c["ref"]} for c in noisy]
t_denoise = time.time() - t0

r_sub = [c["ref"] for c in sub]
d_sub = sum(len(c["audio"]) / 16000 for c in sub)
hn, tn = transcribe(noisy)
hd, td = transcribe(den)

rows = [
    {"condition": "clean", "WER": rate(r_sub, hyp_clean[:N]), "CER": rate(r_sub, hyp_clean[:N], True), "RTF": sum(t_clean[:N]) / d_sub},
    {"condition": "noisy 10 dB", "WER": rate(r_sub, hn), "CER": rate(r_sub, hn, True), "RTF": sum(tn) / d_sub},
    {"condition": "noisy + denoise", "WER": rate(r_sub, hd), "CER": rate(r_sub, hd, True), "RTF": (sum(td) + t_denoise) / d_sub},
]
res = pd.DataFrame(rows)
print(res)
res.to_csv("ablation_results.csv", index=False)

Note: you may need to restart the kernel to use updated packages.
         condition       WER       CER       RTF
0            clean  1.050996  1.065104  0.298102
1      noisy 10 dB  1.037450  1.007750  0.532311
2  noisy + denoise  1.089243  1.013370  0.783205


In [6]:
for i in range(4):
    print("REF:", refs[i])
    print("HYP:", hyp_clean[i])
    print("len ratio:", round(len(norm(hyp_clean[i])) / max(1, len(norm(refs[i]))), 2))
    print()

REF: कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है
HYP:  کچھ عالموں میں آسٹھ کینڈра کوتا ہے جس کا مطلب یہاں کی ان میں تھوڑے یا بھی نہ کسی جھت کے سے ٹھٹھنے کی پرگتی ہوتی ہے
len ratio: 0.98

REF: ग्रीनलैंड को बहुत कम जगह बसाया गया था नॉर्स सगास में वे कहते हैं कि एरिक रेड हत्या के लिए आइसलैंड से निर्वासित किया गया था और आगे पश्चिम की यात्रा करते समय ग्रीनलैंड मिला जिसे ग्रीनलैंड नाम दिया गया
HYP:  Greenland کو بہت کم جگہ بسائے گیا تھا,  نورس سگاس میں بے کہتیں,  کہ ایریک رید حتیا کے لئے آئیس لین سے نرواست کیا گیا تھا  اور آگے پشچم کی آترا کرتے سمیں  Greenland ملا جسے Greenland نام دیا گیا
len ratio: 0.94

REF: ऐसी कोई वैश्विक परिभाषा नहीं है जिसके लिए निर्मित सामान एंटीक होते हैं कुछ कर एजेंसियां 100 साल से पुराने सामान को एंटीक के तौर पर परिभाषित करती हैं
HYP:  Aishi koi waesbik parivasha nahi hai,  Jiske lea nirvith saman antik hote hai,  Kuchkar agenshia, Shao saal se puranish saman ko antik ke tor parivashit karthi

In [7]:
import torch, time
from transformers import WhisperProcessor, WhisperForConditionalGeneration

proc = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
model.generation_config.forced_decoder_ids = None

def transcribe(items):
    outs, times = [], []
    for c in items:
        t = time.time()
        feats = proc(c["audio"], sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
        with torch.no_grad():
            ids = model.generate(feats, language="hi", task="transcribe", max_new_tokens=128, num_beams=1)
        torch.cuda.synchronize()
        times.append(time.time() - t)
        outs.append(proc.batch_decode(ids, skip_special_tokens=True)[0])
    return outs, times

def deva_share(s):
    letters = [ch for ch in s if ch.isalpha()]
    return sum("\u0900" <= ch <= "\u097F" for ch in letters) / max(1, len(letters))

test = clips[:10]
h10, t10 = transcribe(test)
r10 = [c["ref"] for c in test]
for i in range(3):
    print("REF:", r10[i])
    print("HYP:", h10[i])
print("Devanagari share:", np.mean([deva_share(h) for h in h10]))
print("WER on 10:", rate(r10, h10), "CER:", rate(r10, h10, True))

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


REF: कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है
HYP:  अग्वो में आज्द्टर केंद्रख होता है, जिसका मतला भी आजा की उन्मे थोडे या बिना किसी जटके से तुटनें की प्रवत्ती होती है.
REF: ग्रीनलैंड को बहुत कम जगह बसाया गया था नॉर्स सगास में वे कहते हैं कि एरिक रेड हत्या के लिए आइसलैंड से निर्वासित किया गया था और आगे पश्चिम की यात्रा करते समय ग्रीनलैंड मिला जिसे ग्रीनलैंड नाम दिया गया
HYP:  गरीलेंड को बहुत कम जग़़ बसाया गया ता नोर्ष शगास में भे कहते हैं कि एरेक रेड रेद हत्या कि लिए आइस लेंड से निरवासित की आप या �
REF: ऐसी कोई वैश्विक परिभाषा नहीं है जिसके लिए निर्मित सामान एंटीक होते हैं कुछ कर एजेंसियां 100 साल से पुराने सामान को एंटीक के तौर पर परिभाषित करती हैं
HYP:  अएजेई कोई वैस्पिक परिवाशा नहीं है, जिसके लिए निरमिद समान अंटिक होते है, पुचकर अजंश्या स्वाव साल से पुरानि समान को अंटिक
Devanagari share: 1.0
WER on 10: 0.6179775280898876 CER: 0.37683823529411764


In [8]:
%pip install -q noisereduce
import noisereduce as nr

def transcribe(items):
    outs, times = [], []
    for c in items:
        t = time.time()
        feats = proc(c["audio"], sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
        with torch.no_grad():
            ids = model.generate(feats, language="hi", task="transcribe", max_new_tokens=224, num_beams=1)
        torch.cuda.synchronize()
        times.append(time.time() - t)
        outs.append(proc.batch_decode(ids, skip_special_tokens=True)[0])
    return outs, times

# 1) baseline on all 100 clips, language forced to Hindi
hyp_clean, t_clean = transcribe(clips)
dur = sum(len(c["audio"]) / 16000 for c in clips)
print("BASELINE (100 clips) WER", rate(refs, hyp_clean), "CER", rate(refs, hyp_clean, True), "RTF", sum(t_clean) / dur)
pd.DataFrame({"ref": refs, "hyp": hyp_clean, "sec": t_clean}).to_csv("asr_baseline_forced_hi.csv", index=False)

# 2) noisy (10 dB) and denoised, on the first 50 clips
rng = np.random.default_rng(0)
def add_noise(x, snr=10):
    p = np.mean(x ** 2)
    return x + rng.standard_normal(len(x)).astype(np.float32) * np.sqrt(p / 10 ** (snr / 10))

N = 50
sub = clips[:N]
noisy = [{"audio": add_noise(c["audio"]), "ref": c["ref"]} for c in sub]
t0 = time.time()
den = [{"audio": nr.reduce_noise(y=c["audio"], sr=16000).astype(np.float32), "ref": c["ref"]} for c in noisy]
t_den = time.time() - t0

r_sub = [c["ref"] for c in sub]
d_sub = sum(len(c["audio"]) / 16000 for c in sub)
hn, tn = transcribe(noisy)
hd, td = transcribe(den)

rows = [
    {"condition": "clean", "WER": rate(r_sub, hyp_clean[:N]), "CER": rate(r_sub, hyp_clean[:N], True), "RTF": sum(t_clean[:N]) / d_sub},
    {"condition": "noisy 10 dB", "WER": rate(r_sub, hn), "CER": rate(r_sub, hn, True), "RTF": sum(tn) / d_sub},
    {"condition": "noisy + denoise", "WER": rate(r_sub, hd), "CER": rate(r_sub, hd, True), "RTF": (sum(td) + t_den) / d_sub},
]
res = pd.DataFrame(rows)
print(res)
res.to_csv("ablation_results.csv", index=False)

Note: you may need to restart the kernel to use updated packages.
BASELINE (100 clips) WER 0.6711694326514859 CER 0.3798384066140549 RTF 0.22187758784580736
         condition       WER       CER       RTF
0            clean  0.694024  0.407092  0.230382
1      noisy 10 dB  0.795219  0.522379  0.245674
2  noisy + denoise  0.949004  0.650649  0.264127


In [9]:
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

def respond(text):
    msgs = [{"role": "system", "content": "You are a helpful assistant. Reply in Hindi in one or two short sentences."},
            {"role": "user", "content": text}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=80, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def deva_share(s):
    letters = [ch for ch in s if ch.isalpha()]
    return sum("\u0900" <= ch <= "\u097F" for ch in letters) / max(1, len(letters))

def new_numbers(inp, resp):
    nums = re.findall(r"[0-9\u0966-\u096F]+", resp)
    return any(n not in inp for n in nums)

def f1(a, b):
    a, b = norm(a).split(), norm(b).split()
    common = sum(min(a.count(w), b.count(w)) for w in set(a))
    if not a or not b or common == 0:
        return 0.0
    p, r = common / len(a), common / len(b)
    return 2 * p * r / (p + r)

respond(hyp_clean[0])  # warm-up, not timed
resp, t_llm = [], []
for h in hyp_clean:
    t = time.time()
    resp.append(respond(h))
    torch.cuda.synchronize()
    t_llm.append(time.time() - t)

total = np.array(t_clean) + np.array(t_llm)
ORC = 50
resp_oracle = [respond(r) for r in refs[:ORC]]
agree = [f1(resp[i], resp_oracle[i]) for i in range(ORC)]

df = pd.DataFrame({"ref": refs, "asr": hyp_clean, "response": resp,
                   "hallu_number": [new_numbers(h, r) for h, r in zip(hyp_clean, resp)],
                   "wrong_script": [deva_share(r) < 0.5 for r in resp],
                   "asr_s": t_clean, "llm_s": t_llm, "total_s": total})
df["manual_score_0_1_2"] = ""
df.to_csv("pipeline_responses.csv", index=False)
df.sample(20, random_state=0).to_csv("manual_scoring_sample.csv", index=False)

print("hallucination (new number) rate:", df.hallu_number.mean())
print("wrong-script response rate:", df.wrong_script.mean())
print("response agreement ASR vs reference input (F1, first 50):", np.mean(agree))
print("LLM mean s:", np.mean(t_llm), "| total mean s:", total.mean(), "| p95 s:", np.percentile(total, 95))
print("pipeline RTF:", total.sum() / dur)

tokenizer_config.json: 0.00B [00:00, ?B/s]

E:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sankalp\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

E:\Anaconda\Lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
E:\Anaconda\Lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
E:\Anaconda\Lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


hallucination (new number) rate: 0.02
wrong-script response rate: 0.0
response agreement ASR vs reference input (F1, first 50): 0.1976647598609429
LLM mean s: 4.336459984779358 | total mean s: 7.039328384399414 | p95 s: 8.830140793323517
pipeline RTF: 0.5778561776091722


In [11]:
import torch

# Remove generation warnings
llm.generation_config.temperature = None
llm.generation_config.top_p = None
llm.generation_config.top_k = None

def respond(text):
    msgs = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Reply in Hindi in one or two short sentences."
        },
        {
            "role": "user",
            "content": text
        }
    ]

    # Create enc inside the function
    enc = tok.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    with torch.inference_mode():
        out = llm.generate(
            **enc,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    response = tok.decode(
        out[0][enc["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return response

In [12]:
test = respond("भारत की राजधानी क्या है?")
print(test)

भारत की राजधानी गुजरात सिंहदल केंद्र है।


In [13]:
print("Test response:", respond("भारत की राजधानी क्या है?"))

Test response: भारत की राजधानी गुजरात सिंहदल केंद्र है।


In [14]:
import os
import pandas as pd

print("Working directory:", os.getcwd())

for filename in [
    "pipeline_responses.csv",
    "manual_scoring_sample.csv"
]:
    if os.path.exists(filename):
        df = pd.read_csv(filename)
        print(f"{filename}: FOUND — {len(df)} rows")
    else:
        print(f"{filename}: NOT FOUND")

Working directory: C:\Users\Sankalp
pipeline_responses.csv: FOUND — 100 rows
manual_scoring_sample.csv: FOUND — 20 rows


In [15]:
import pandas as pd

df = pd.read_csv("manual_scoring_sample.csv")

pd.set_option("display.max_colwidth", 150)

display(df[["ref", "asr", "response"]])

,ref,asr,response
0,उन्होंने अफवाहों को राजनीतिक बकवास और मूर्खतापूर्ण कहा,उंहुन आना खवाओ कराजनी तिक बखवास उर्मोग क्तपन का है.,उंहुन आना खवाओ कराजनी तिक बखवास उर्मोग क्तपन का है।\n\nयह समस्या एक महत्वपूर्ण रोग है ज
1,वहीं स्पेक्ट्रम का दूसरा सिरा कोई व्यक्ति किसी ऐसे अपिरिचित व्यक्ति का रूप लेता है जिसे लगता है कि उसे टीम में सब कुछ बदल देना चाहिए और उसे अपने ह...,वही विस्प्रेट्रम का दुस्रा सिरा कोई वेख्ती किसी अज़े अपरिचद वेख्ती का रूप लेता है जिसे लगता है कि उसर टींग में सब कुई बड़ल देना जाहीए और उसर अपने...,"वही विस्प्रेट्रम का दुसरा सिरा एक अज़े अपरिचद वेख्ती का रूप लेता है, जिसे लगता है कि"
2,ऐसी कोई वैश्विक परिभाषा नहीं है जिसके लिए निर्मित सामान एंटीक होते हैं कुछ कर एजेंसियां 100 साल से पुराने सामान को एंटीक के तौर पर परिभाषित करती हैं,"अएजेई कोई वैस्पिक परिवाशा नहीं है, जिसके लिए निरमिद समान अंटिक होते है, पुचकर अजंश्या स्वाव साल से पुरानि समान को अंटिक के तोर पर परिवाशित करती है.",अएजेई कोई वैस्पिक परिवाशा नहीं है।
3,चीज़ों पर कुछ बनाकर या खरोंच कर इस जगह को नुक़सान न पहुंचाएं,तो थो थो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो त...,यह प्रत्यक्ष संदेश है। इसे अपनी मांग करना और उसे बढ़ावा देना चाहिए।
4,अटलांटिक तूफ़ान मौसम का दसवां तूफ़ान उपोष्णकटिबंधीय तूफ़ान जेरी आज अटलांटिक महासागर में बना है,अट्लान्टिक तुफान मुसम का दस्वा तुफान उपोषन कर्टिबंदे तुफान जेरी आज अट्लान्टिक माशागर में बना है,अटलांटिक तुफान मुसम का दस्वा तुफान उपोषन कर्टिबंदे तुफान जेरी आज अट्लांटिक माशागर में
5,जियानकार्लो फिशिचेला ने अपनी कार पर नियंत्रण खो दिया और शुरू करने के कुछ समय बाद ही दौड़ समाप्त कर दी,जीयान कारलो पशी चेला ने अपनी कार पर नियन्द्रन खो दिया और शुरू करने के कुछ समें बादी तोर समापत कर दी,जीयान कारलो ने अपनी कार पर नियंद्रण की आवश्यकता को विकसित कर दी और इसके बाद उसे शुरू कर
6,गंभीर मौसम किसी भी खतरनाक मौसमी घटना के लिए वर्गीय शब्द है जिसमें नुकसान गंभीर सामाजिक व्यवधान या मानव जीवन की हानि की संभावना है,"गंबीर मुसम किसी भी खटनाक मुसम लिए वर्गी ये सब दै, जिस में नुक्सान गंबीर सामाजिक वेवदान या मानव जीवन की हानी की संबावना है.",गंबीर मुसम किसी भी खटनाक मुसम लिए वर्गीय अधिकारों की आवश्यकता है। इसमें नुक्सान गंबी
7,हालांकि यह अकेला नहीं है प्रयोग करना और प्रयोग एक ऐसा परीक्षण होता है जिसका इस्तेमाल करके एक या इससे ज़्यादा संभावित परिकल्पनाओं को ख़त्म करने सवा...,आँगा कि यह अगेला नहीं है परिएुक करना अपे एक एसा परिक्सन होता है जिसका इस्तमाल करके एक यह यस जियादा संबवित परिकल्पनावो को कतम करनेक नहीं सवाल करने...,"यह बात भी शामिल है। यह अगेला नहीं है, और उसे रखना चाहिए। यह एक ऐसा प्रकार का विकास �"
8,एक सामान्य लोकप्रिय डिनर विशेष रूप से गर्मी के मौसम में पा एम्बे ओली है ब्रेड के साथ ऑलिव ऑयल टमाटर और कोई भी उपलब्ध रुचिकर सामग्री जैसे पनीर ट्यू...,"एक सामाने लोक प्रियर दिनर विशेश रूप से गर्मी के माँसम में पा आमभे ओली है ब्रेट के साद अलीवायल, तमातर और कोई भी उपलब्द रुचिकर सामगरी ज़ैसे पनीर च्...",एक सामाने लोक प्रियर दिनर विशेष रूप से गर्मी के माँसम में पा आमभे ओली है। ब्रेट के साद अली
9,1889 में यह बंदरगाह कुख्यात नौसैनिक गतिरोध का ठिकाना था उस समय जर्मनी अमेरिका और ब्रिटेन के सात जहाजों ने इस बंदरगाह से जाने से इनकार कर दिया था,अपन्दर्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा...,"यह एक वाक्य है, जो कि आपके संस्कृत में लिखी गई है। इसमें बहुत चले रहे शब्द ""प्रद्गा"""


In [16]:
import pandas as pd
import re
from collections import Counter

df = pd.read_csv("pipeline_responses.csv")

def transcript_quality(text):
    words = re.findall(r"\S+", str(text))
    if not words:
        return {
            "unique_ratio": 0.0,
            "max_repetition": 0,
            "flagged": True
        }

    counts = Counter(words)
    unique_ratio = len(counts) / len(words)
    max_repetition = max(counts.values())

    # Heuristic thresholds: tune on a separate development set
    flagged = (
        (len(words) >= 10 and unique_ratio < 0.35)
        or max_repetition >= 8
    )

    return {
        "unique_ratio": unique_ratio,
        "max_repetition": max_repetition,
        "flagged": flagged
    }

quality = df["asr"].apply(transcript_quality)

df["unique_ratio"] = quality.apply(
    lambda x: x["unique_ratio"]
)
df["max_repetition"] = quality.apply(
    lambda x: x["max_repetition"]
)
df["quality_flag"] = quality.apply(
    lambda x: x["flagged"]
)

print("Total samples:", len(df))
print("Flagged transcripts:", int(df["quality_flag"].sum()))
print("Flag rate:", round(df["quality_flag"].mean(), 4))

display(
    df[df["quality_flag"]][
        ["ref", "asr", "response", "unique_ratio", "max_repetition"]
    ].head(20)
)

df.to_csv("transcript_quality_analysis.csv", index=False)

Total samples: 100
Flagged transcripts: 8
Flag rate: 0.08


,ref,asr,response,unique_ratio,max_repetition
17,हालांकि यह अकेला नहीं है प्रयोग करना और प्रयोग एक ऐसा परीक्षण होता है जिसका इस्तेमाल करके एक या इससे ज़्यादा संभावित परिकल्पनाओं को ख़त्म करने सवा...,अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अ...,"यह बात कुछ मिलकर आती है। यदि आप इसे जल्द समझना चाहते हैं, तो उसे एक-एक प्रकार के विशेष",0.044444,44
33,ऐसा माना जाता है कि पेरिस के निवासी अहंकारी असभ्य और अभिमानी होते हैं,आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आ...,"मैं अभी तक कुछ नहीं सुना हूँ। यदि आपको मामले किसी भी प्रकार की सवाल है, तो मैं उसे सुधा�",0.051724,52
34,1963 में बांध बनने के बाद मौसमी बाढ़ जो पूरे नदी में तलछट फैला देगी वह रुक गई थी,अगर तो बद्बाद बाद बद्बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद बाद ...,यह प्रकट हुआ कि आपका संस्कृतिशीलता और व्यवहार में निष्पक्ष भावनाएँ हैं। इसे जागरूक �,0.113636,39
45,स्प्रिंगबोक्स के लिए इसने पांच-मैचों की हार का सिलसिला समाप्त कर दिया,अपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने आपने ...,यह प्रकाशित करता है कि आपको एक बड़ी संख्या दिखाना चाहिए।,0.078947,36
46,लोग अब कंप्यूटर स्क्रीन पर संदेश लिखते हैं शार्पनर का उपयोग करने की आवश्यकता नहीं पड़ती,लोग आप ख़ब ख़ब श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वाद श्वा...,"मैं अभी तक कुछ भी नहीं सुना हूँ। यदि आपके बारे में जानकारी चाहते हैं, तो इस प्रकार �",0.166667,25
55,चीज़ों पर कुछ बनाकर या खरोंच कर इस जगह को नुक़सान न पहुंचाएं,तो थो थो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो त...,यह प्रत्यक्ष संदेश है। इसे अपनी मांग करना और उसे बढ़ावा देना चाहिए।,0.040000,72
82,यह संबंधित है लेकिन आम तौर पर इसमें अल्पाइन शैली की स्की टूरिंग या माउंटेनीयरिंग शामिल नहीं होती है जिसे खड़े इलाकों में किया जाता है और जिसके लिए...,अपने अपने लिए प्रदी तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए तो लिए...,"यह संक्षेप में कुछ वाक्य हैं, जो आपका निर्णय दर्शाते हैं। इसमें आपका उत्सव और खेल के",0.074074,26
95,1889 में यह बंदरगाह कुख्यात नौसैनिक गतिरोध का ठिकाना था उस समय जर्मनी अमेरिका और ब्रिटेन के सात जहाजों ने इस बंदरगाह से जाने से इनकार कर दिया था,अपन्दर्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा प्रद्गा...,"यह एक वाक्य है, जो कि आपके संस्कृत में लिखी गई है। इसमें बहुत चले रहे शब्द ""प्रद्गा""",0.130435,21


In [17]:
import pandas as pd

df = pd.read_csv("transcript_quality_analysis.csv")

clarification = (
    "क्षमा करें, आपकी बात स्पष्ट रूप से समझ नहीं आई। "
    "कृपया दोबारा बोलें।"
)

df["improved_response"] = df["response"].copy()

df.loc[
    df["quality_flag"], "improved_response"
] = clarification

df["action"] = df["quality_flag"].map({
    True: "clarification",
    False: "respond"
})

print("Total:", len(df))
print("Clarifications:", (df["action"] == "clarification").sum())
print("Normal responses:", (df["action"] == "respond").sum())

df.to_csv(
    "improved_pipeline_responses.csv",
    index=False,
    encoding="utf-8-sig"
)

Total: 100
Clarifications: 8
Normal responses: 92


In [18]:
import pandas as pd

df = pd.read_csv("transcript_quality_analysis.csv")

experiments = {
    "Baseline (no gate)": pd.Series(False, index=df.index),

    "Repetition only": df["max_repetition"] >= 8,

    "Lexical diversity only": (
        (df["unique_ratio"] < 0.35) &
        (df["asr"].fillna("").str.split().str.len() >= 10)
    ),

    "Combined gate": df["quality_flag"].astype(bool)
}

results = []

for name, flagged in experiments.items():
    results.append({
        "configuration": name,
        "flagged_samples": int(flagged.sum()),
        "normal_responses": int((~flagged).sum()),
        "clarification_rate": round(float(flagged.mean()), 4)
    })

ablation = pd.DataFrame(results)

display(ablation)

ablation.to_csv(
    "quality_gate_ablation.csv",
    index=False
)

,configuration,flagged_samples,normal_responses,clarification_rate
0,Baseline (no gate),0,100,0.00
1,Repetition only,8,92,0.08
2,Lexical diversity only,8,92,0.08
3,Combined gate,8,92,0.08


In [19]:
import pandas as pd

df = pd.read_csv("transcript_quality_analysis.csv")

repetition = df["max_repetition"] >= 8

diversity = (
    (df["unique_ratio"] < 0.35) &
    (df["asr"].fillna("").str.split().str.len() >= 10)
)

print("Repetition flagged:", repetition.sum())
print("Diversity flagged:", diversity.sum())
print("Flagged by both:", (repetition & diversity).sum())
print("Repetition only:", (repetition & ~diversity).sum())
print("Diversity only:", (~repetition & diversity).sum())
print("Identical selections:", repetition.equals(diversity))

Repetition flagged: 8
Diversity flagged: 8
Flagged by both: 8
Repetition only: 0
Diversity only: 0
Identical selections: True


In [20]:
import pandas as pd

df = pd.read_csv("transcript_quality_analysis.csv")

# Recover boolean flag safely
flag = df["quality_flag"].astype(str).str.lower().eq("true")

flagged = df[flag]
unflagged = df[~flag].sample(
    n=min(20, (~flag).sum()),
    random_state=42
)

review = pd.concat([flagged, unflagged]).sample(
    frac=1, random_state=42
).copy()

# Manual annotation fields
review["asr_unusable_0_1"] = ""
review["response_quality_0_1_2"] = ""
review["unsupported_claim_0_1"] = ""
review["notes"] = ""

review.to_csv(
    "task2_manual_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total to review:", len(review))
print("Flagged examples:", len(flagged))
print("Unflagged examples:", len(unflagged))
print("Saved: task2_manual_review.csv")

Total to review: 28
Flagged examples: 8
Unflagged examples: 20
Saved: task2_manual_review.csv


In [21]:
import pandas as pd

df = pd.read_csv("task2_manual_review.csv")

cols = [
    "asr_unusable_0_1",
    "response_quality_0_1_2",
    "unsupported_claim_0_1"
]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Total examples:", len(df))
print("Fully labeled:", df[cols].notna().all(axis=1).sum())
print("Still incomplete:", (~df[cols].notna().all(axis=1)).sum())

Total examples: 28
Fully labeled: 0
Still incomplete: 28
